# **Contexto**  
Este conjunto de datos tiene 15,150 imágenes de 12 clases diferentes de basura doméstica: papel, cartón, desechos orgánicos, metal, plástico, vidrio verde, vidrio café, vidrio blanco, ropa, zapatos, baterías y basura general.

El reciclaje de basura es un aspecto clave para preservar nuestro medio ambiente. Para hacer posible o facilitar el proceso de reciclaje, la basura debe clasificarse en grupos que tengan un proceso de reciclaje similar. Noté que la mayoría de los conjuntos de datos disponibles clasifican la basura en pocas categorías (entre 2 y 6 clases como máximo). Tener la capacidad de clasificar la basura doméstica en más clases puede resultar en un incremento significativo en el porcentaje de basura reciclada.

# **Contenido**  
Un escenario ideal para la recolección de datos sería colocar una cámara sobre una cinta transportadora donde la basura pase una por una, de modo que la cámara pueda capturar imágenes reales de los desechos. Pero como ese tipo de configuración no es posible por el momento, la mayoría de las imágenes de este conjunto de datos fueron recopiladas mediante web scraping. Traté de obtener imágenes lo más parecidas posible a basura real siempre que fue posible; por ejemplo, para la categoría de desechos biológicos busqué vegetales podridos, frutas descompuestas y restos de comida, etc.

Sin embargo, para algunas clases como ropa o zapatos fue más difícil obtener imágenes de estos objetos en la basura, por lo que en su mayoría fueron imágenes de ropa normal. Aun así, poder clasificar las imágenes de este conjunto de datos en 12 clases puede ser un gran paso para mejorar el proceso de reciclaje.

In [ ]:
# 1. Instalar librerías básicas
%pip install -q pandas numpy matplotlib scikit-learn jupyter

# 2. Instalar PyTorch y el driver DirectML (el traductor de AMD para que PyTorch funcione con la GPU chafa)
%pip install torch torchvision torch-directml
print("Instalación completada.")

In [ ]:
%pip uninstall -y torch torchvision torch-directml

In [ ]:
# Instalar PyTorch 2.2.0 y su complemento DirectML compatible
%pip install torch==2.2.0 torchvision==0.17.0 torch-directml

In [ ]:
%pip install "numpy<2.0"

In [ ]:
# 1. Borrar todo rastro de las versiones actuales
%pip uninstall -y torch torchvision torch-directml numpy

In [ ]:
# 2. Instalar las versiones específicas que no dan error de DLL
# (Numpy debe ser menor a 2.0 para esta versión de Torch)
%pip install torch==1.13.1 torchvision==0.14.1 torch-directml "numpy<2.0" protobuf==3.20.0

In [ ]:
import torch
import torch_directml

device = torch_directml.device()

# 1. Crear dato en GPU
x = torch.tensor([5.0]).to(device)
y = torch.tensor([10.0]).to(device)

# 2. Hacer una operación (para probar que calcula)
resultado = x + y

# 3. Imprimir CORRECTAMENTE (usando .item() para sacar el número)
print(f" GPU Conectada. Resultado de la suma: {resultado.item()}")

**Librerias**

In [ ]:
import os
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

# --- Para AMD ---
import torch_directml  
# -------------------

**Configuracion para que las imagenes se vean bien**

In [ ]:
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

In [ ]:
BASE_DIR = "garbage_classification"

In [ ]:
print("Versión de torch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))



In [ ]:
device = torch_directml.device()
print(f"GPU AMD Conectada. Dispositivo: {device}")

**Instalar paquetes**

**Exploracion de estructura y clases del cojunto de datos**

In [ ]:
classes = sorted([
    d for d in os.listdir(BASE_DIR)
    if os.path.isdir(os.path.join(BASE_DIR, d))
])

print(f"Número de clases: {len(classes)}")
print("Clases:", classes)

**Construir un DataFrame con todas las imágenes**

In [ ]:
data = []

for cls in classes:
    class_dir = os.path.join(BASE_DIR, cls)
    for fname in os.listdir(class_dir):
        if fname.lower().endswith((".jpg", ".jpeg", ".png")):
            full_path = os.path.join(class_dir, fname)
            data.append((full_path, cls))

df = pd.DataFrame(data, columns=["filepath", "label"])

print("Número total de imágenes:", len(df))
df.head()

**Distribución de clases**

In [ ]:
class_counts = df["label"].value_counts().sort_index()
class_percent = (class_counts / len(df) * 100).round(2)

dist_df = pd.DataFrame({
    "count": class_counts,
    "percent": class_percent
})

print("Distribución de clases:")
display(dist_df)

In [ ]:
plt.figure()
plt.bar(dist_df.index, dist_df["count"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Número de imágenes")
plt.title("Distribución de imágenes por clase")
plt.tight_layout()
plt.show()

**Tamaños de imágenes**

In [ ]:
widths = []
heights = []

filepaths = df["filepath"].tolist()

import random
filepaths = random.sample(filepaths, 500) #Como son muchas imagenes entonces solo veresos 500

for path in filepaths:
    try:
        with Image.open(path) as img:
            w, h = img.size
            widths.append(w)
            heights.append(h)
    except Exception as e:
        print("Error al abrir:", path, "->", e)

print(f"Imágenes analizadas: {len(widths)}")

def summarize(vals, name):
    print(f"\n{name}:")
    print(f"  Mínimo: {min(vals)}")
    print(f"  Máximo: {max(vals)}")
    print(f"  Media:  {np.mean(vals):.2f}")
    print(f"  Mediana:{np.median(vals):.2f}")

summarize(widths, "Ancho")
summarize(heights, "Alto")

In [ ]:
plt.figure()
plt.hist(widths, bins=20)
plt.xlabel("Ancho (px)")
plt.ylabel("Frecuencia")
plt.title("Distribución del ancho de las imágenes")
plt.show()

plt.figure()
plt.hist(heights, bins=20)
plt.xlabel("Alto (px)")
plt.ylabel("Frecuencia")
plt.title("Distribución del alto de las imágenes")
plt.show()

**Modos de color**

In [ ]:
from collections import Counter

color_modes = Counter()


In [ ]:
from collections import Counter

color_modes = Counter() 
sample_paths = filepaths[:500] 

for path in sample_paths:
    try:
        with Image.open(path) as img:
            color_modes[img.mode] += 1
    except:
        pass

print("Modos de color en una muestra de imágenes:")
for mode, count in color_modes.items():
    print(f"{mode}: {count}")

**Ver ejemplos de imágenes por clase**

In [ ]:
n_per_class = 4  # Solo veremos 4 imágenes por clase

for cls in classes:
    subset = df[df["label"] == cls]["filepath"].tolist()
    if len(subset) == 0:
        continue

    sample = random.sample(subset, min(n_per_class, len(subset)))

    plt.figure(figsize=(10, 3))
    plt.suptitle(f"Ejemplos de clase: {cls}")
    
    for i, path in enumerate(sample, 1):
        try:
            img = Image.open(path)
            plt.subplot(1, n_per_class, i)
            plt.imshow(img)
            plt.axis("off")
        except:
            pass

    plt.tight_layout()
    plt.show()

# **Proyecto**  
### **Clasificación automática de residuos con aprendizaje profundo para apoyar el reciclaje**  
**Objetivo:** Entrenar un modelo que clasifique imágenes en las clases del dataset (battery, biological, cardboard, clothes, glass, metal, plastic, trash, etc.) y analizar qué técnicas ayudan más dado que el dataset está desbalanceado.  


In [ ]:
from sklearn.model_selection import train_test_split

**Preprocesamiento**

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
RANDOM_STATE = 42

**Split estratificado 70/15/15**

In [ ]:
# 1) Train (70%) vs temp (30%)
df_train, df_temp = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=RANDOM_STATE
)

# 2) De ese 30%, sacamos 15% val y 15% test (es decir, mitad y mitad del 30%)
df_val, df_test = train_test_split(
    df_temp,
    test_size=0.50,  # 50% de 30% = 15% del total
    stratify=df_temp["label"],
    random_state=RANDOM_STATE
)

# Resetear índices
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

print("Tamaños finales:")
print("Train:", len(df_train))
print("Val:  ", len(df_val))
print("Test: ", len(df_test))

**Verificar distribución de clases en cada split**

In [ ]:
def show_class_distribution(name, df_split):
    dist = df_split["label"].value_counts(normalize=True).sort_index() * 100
    print(f"\n{name} - distribución (%)")
    print(dist.round(2))

show_class_distribution("TRAIN", df_train)
show_class_distribution("VAL",   df_val)
show_class_distribution("TEST",  df_test)

**Definir transforms (redimensionar + normalizar)**  
Usamos las medias y desviaciones estándar típicas de ImageNet para normalizar (sirve muy bien con modelos preentrenados).

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],  # valores típicos de ImageNet
        std=[0.229, 0.224, 0.225]
    )
])

val_test_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


**Dataset personalizado de PyTorch**

In [ ]:
class GarbageDataset(Dataset):
    def __init__(self, df_split, transform=None, class_to_idx=None):
        self.df = df_split
        self.transform = transform
        
        # Mapear etiquetas string -> índice numérico
        if class_to_idx is None:
            classes = sorted(self.df["label"].unique())
            self.class_to_idx = {cls: i for i, cls in enumerate(classes)}
        else:
            self.class_to_idx = class_to_idx

        # Para referencia inversa si la necesitas luego
        self.idx_to_class = {v: k for k, v in self.class_to_idx.items()}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row["filepath"]
        label_str = row["label"]
        label = self.class_to_idx[label_str]

        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        return image, label

**Crear Datasets y DataLoaders**

In [ ]:
# Creamos un solo mapeo de clases para usar el mismo en train/val/test
classes = sorted(df["label"].unique())
class_to_idx = {cls: i for i, cls in enumerate(classes)}
print("class_to_idx:", class_to_idx)

train_dataset = GarbageDataset(df_train, transform=train_transform,     class_to_idx=class_to_idx)
val_dataset   = GarbageDataset(df_val,   transform=val_test_transform, class_to_idx=class_to_idx)
test_dataset  = GarbageDataset(df_test,  transform=val_test_transform, class_to_idx=class_to_idx)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True        
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True        
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True        
)

print("Batches por época:")
print("Train:", len(train_loader))
print("Val:  ", len(val_loader))
print("Test: ", len(test_loader))


In [ ]:
import time

In [ ]:
t0 = time.time()
images, labels = next(iter(train_loader))
t1 = time.time()

print("Tiempo para el primer batch:", round(t1 - t0, 2), "segundos")
print("Shape del batch de imágenes:", images.shape)  # [batch_size, 3, 224, 224]
print("Shape del batch de labels:", labels.shape)
print("Primeros labels:", labels[:10])

**Métricas, device y funciones genéricas de train/eval**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import torch_directml  # <--- IMPORTANTE: Librería para AMD
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# --- CONFIGURACIÓN GPU AMD ---
# Aquí le decimos que use el driver DirectML
device = torch_directml.device()
print(f" Usando device: {device}")
# -----------------------------

num_classes = len(class_to_idx)
idx_to_class = {v: k for k, v in class_to_idx.items()}


def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_labels = []
    all_preds = []

    for images, labels in dataloader:
        # Enviamos datos a la RX 9060 XT
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = outputs.argmax(dim=1)
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average="macro")

    return epoch_loss, epoch_acc, epoch_f1


def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in dataloader:
            
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average="macro")

    return epoch_loss, epoch_acc, epoch_f1, np.array(all_labels), np.array(all_preds)


def train_model(model, train_loader, val_loader, criterion, optimizer, device, epochs=10):
    
    # Enviamos el modelo completo a la GPU
    model = model.to(device)

    history = {
        "train_loss": [],
        "train_acc": [],
        "train_f1": [],
        "val_loss": [],
        "val_acc": [],
        "val_f1": [],
    }

    best_val_f1 = 0.0
    best_state_dict = None

    print(" Iniciando entrenamiento en GPU AMD...")

    for epoch in range(1, epochs + 1):
        train_loss, train_acc, train_f1 = train_one_epoch(
            model, train_loader, criterion, optimizer, device
        )
        val_loss, val_acc, val_f1, _, _ = evaluate(
            model, val_loader, criterion, device
        )

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["train_f1"].append(train_f1)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["val_f1"].append(val_f1)

        print(
            f"Epoch [{epoch}/{epochs}] "
            f"Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f} || "
            f"Val Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f}"
        )

        # Guardar el mejor modelo según F1-macro en validación
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state_dict = model.state_dict()

    # Cargar el mejor estado al final
    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)
        print(" Entrenamiento finalizado. Se cargó el mejor modelo.")

    return model, history


def plot_confusion_matrix(y_true, y_pred, idx_to_class, title="Matriz de confusión"):
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=[idx_to_class[i] for i in range(len(idx_to_class))]
    )
    # Aumenté un poco el tamaño para que se vea mejor
    fig, ax = plt.subplots(figsize=(10, 10))
    disp.plot(ax=ax, xticks_rotation=90)
    plt.title(title)
    plt.tight_layout()
    plt.show()

## **Modelo 1 – CNN Baseline sencilla**

**Entrenar la CNN baseline**

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()

        self.features = nn.Sequential(
            # Bloque 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 224 -> 112

            # Bloque 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 112 -> 56

            # Bloque 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 56 -> 28
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


# Crear modelo baseline
baseline_model = SimpleCNN(num_classes=num_classes)

# Opcional: solo para ver que se mueve a GPU
baseline_model = baseline_model.to(device)

print(baseline_model)


In [ ]:
criterion_baseline = nn.CrossEntropyLoss()
optimizer_baseline = optim.Adam(baseline_model.parameters(), lr=1e-3)

EPOCHS_BASELINE = 10

baseline_model, hist_baseline = train_model(
    baseline_model,
    train_loader,
    val_loader,
    criterion_baseline,
    optimizer_baseline,
    device,              
    epochs=EPOCHS_BASELINE
)


**Evaluar baseline en el conjunto de prueba**

In [ ]:
test_loss_b, test_acc_b, test_f1_b, y_true_b, y_pred_b = evaluate(
    baseline_model, test_loader, criterion_baseline, device
)

print(f"Baseline - Test Loss: {test_loss_b:.4f}")
print(f"Baseline - Test Accuracy: {test_acc_b:.4f}")
print(f"Baseline - Test F1-macro: {test_f1_b:.4f}")

print("\nReporte de clasificación (Baseline):")
print(classification_report(y_true_b, y_pred_b, target_names=[idx_to_class[i] for i in range(num_classes)]))

plot_confusion_matrix(y_true_b, y_pred_b, idx_to_class, title="Matriz de confusión - Baseline CNN")


**Transfer Learning**  
Usamos ResNet18 preentrenada en ImageNet, congelamos capas y solo entrenamos la parte final.

In [ ]:
import torchvision.models as models

In [ ]:
resnet = models.resnet18(pretrained=True)


for param in resnet.parameters():
    param.requires_grad = False


in_features = resnet.fc.in_features
resnet.fc = nn.Linear(in_features, num_classes)

resnet = resnet.to(device)
print(resnet.fc)


**Entrenar modelo de Transfer Learning**  
Aquí solo se entrenan los pesos de la última capa.

In [ ]:
criterion_resnet = nn.CrossEntropyLoss()
optimizer_resnet = optim.Adam(resnet.fc.parameters(), lr=1e-3)

EPOCHS_RESNET = 8

resnet, hist_resnet = train_model(
    resnet,
    train_loader,
    val_loader,
    criterion_resnet,
    optimizer_resnet,
    device,
    epochs=EPOCHS_RESNET
)


In [ ]:
test_loss_r, test_acc_r, test_f1_r, y_true_r, y_pred_r = evaluate(
    resnet, test_loader, criterion_resnet, device
)

print(f"ResNet18 - Test Loss: {test_loss_r:.4f}")
print(f"ResNet18 - Test Accuracy: {test_acc_r:.4f}")
print(f"ResNet18 - Test F1-macro: {test_f1_r:.4f}")

print("\nReporte de clasificación (ResNet18):")
print(classification_report(y_true_r, y_pred_r, target_names=[idx_to_class[i] for i in range(num_classes)]))

plot_confusion_matrix(y_true_r, y_pred_r, idx_to_class, title="Matriz de confusión - ResNet18")


## **Modelo 3 – Manejo de desbalance (pesos + data augmentation)**  
Aquí tomamos el mejor modelo y:

Calculamos class_weight a partir de df_train.

Usamos un CrossEntropyLoss(weight=...).

Metemos data augmentation al train_transform.

**Calcular pesos por clase**

In [ ]:
class_counts = df_train["label"].value_counts().sort_index()
print("Conteos por clase en TRAIN:")
print(class_counts)

class_weights = []
for cls in classes:
    count = class_counts[cls]      
    weight = 1.0 / count           
    class_weights.append(weight)

class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)


**Nuevos transforms con data augmentation para TRAIN**

In [ ]:
aug_train_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Validación y test sin augmentation (solo resize + normalize)
plain_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset_bal = GarbageDataset(df_train, transform=aug_train_transform, class_to_idx=class_to_idx)
val_dataset_bal   = GarbageDataset(df_val,   transform=plain_transform,   class_to_idx=class_to_idx)
test_dataset_bal  = GarbageDataset(df_test,  transform=plain_transform,   class_to_idx=class_to_idx)

train_loader_bal = DataLoader(
    train_dataset_bal,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)
val_loader_bal = DataLoader(
    val_dataset_bal,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)
test_loader_bal = DataLoader(
    test_dataset_bal,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("Batches por época (con augmentation):")
print("Train:", len(train_loader_bal))
print("Val:  ", len(val_loader_bal))
print("Test: ", len(test_loader_bal))


**Nuevo modelo ResNet18 con class_weight**

In [ ]:
resnet_bal = models.resnet18(pretrained=True)


for param in resnet_bal.parameters():
    param.requires_grad = False


in_features_bal = resnet_bal.fc.in_features
resnet_bal.fc = nn.Linear(in_features_bal, num_classes)  # nueva capa, requiere_grad=True por defecto


resnet_bal = resnet_bal.to(device)


criterion_bal = nn.CrossEntropyLoss(weight=class_weights)


optimizer_bal = optim.Adam(resnet_bal.fc.parameters(), lr=1e-3)

EPOCHS_BAL = 8  


resnet_bal, hist_bal = train_model(
    resnet_bal,
    train_loader_bal,
    val_loader_bal,
    criterion_bal,
    optimizer_bal,
    device,
    epochs=EPOCHS_BAL
)

**Evaluar el modelo con manejo de desbalance**

In [ ]:
test_loss_bal, test_acc_bal, test_f1_bal, y_true_bal, y_pred_bal = evaluate(
    resnet_bal, test_loader_bal, criterion_bal, device
)

print(f"ResNet18 + class_weight + augmentation - Test Loss: {test_loss_bal:.4f}")
print(f"ResNet18 + class_weight + augmentation - Test Accuracy: {test_acc_bal:.4f}")
print(f"ResNet18 + class_weight + augmentation - Test F1-macro: {test_f1_bal:.4f}")

print("\nReporte de clasificación (ResNet18 balanceado):")
print(classification_report(y_true_bal, y_pred_bal, target_names=[idx_to_class[i] for i in range(num_classes)]))

plot_confusion_matrix(
    y_true_bal,
    y_pred_bal,
    idx_to_class,
    title="Matriz de confusión - ResNet18 (class_weight + augmentation)"
)

In [ ]:
# --- GUARDAR MODELO ---
# Se guardarán los pesos (state_dict), que es lo estándar, bajo el criterio de 
nombre_archivo = "modelo_basura_resnet18_amd.pth"
torch.save(resnet_bal.state_dict(), nombre_archivo)

print(f"Modelo guardado exitosamente como '{nombre_archivo}'")
print("Se guarada el modelo para no tener que ejecutar todo desde el inicio.")

In [ ]:
import torch
import torch_directml
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt

# --- CONFIGURACIÓN ---
# Definimos las clases tal cual usadas en el entrenamiento
CLASSES = [
    'paper', 'cardboard', 'plastic', 'metal', 'trash', 'battery',
    'shoes', 'clothes', 'green-glass', 'brown-glass', 'white-glass', 'biological'
]
MODEL_PATH = "modelo_basura_resnet18_amd.pth"

# 1. PREPARAR EL DISPOSITIVO (AMD)
# Si se usa en otra PC sin AMD, se puede cambiar a "cpu" para que funcione igual
try:
    device = torch_directml.device()
except:
    device = torch.device("cpu")
print(f"Usando dispositivo: {device}")

# 2. CARGAR EL MODELO
def cargar_modelo():
    # Creamos la estructura vacía
    model = models.resnet18(pretrained=False) # No necesitamos pretrained de internet, usaremos el nuestro ya entrenado
    num_ftrs = model.fc.in_features
    model.fc = torch.nn.Linear(num_ftrs, len(CLASSES))
    
    # Cargamos los pesos guardados
    # map_location asegura que cargue bien aunque cambies de GPU a CPU
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    
    model = model.to(device)
    model.eval() # IMPORTANTE: Poner en modo evaluación (congela dropout, batchnorm, etc.)
    return model

# 3. TRANSFORMACIONES (La clave para que funcione)
# Deben ser idénticas a las de VALIDACIÓN de tu entrenamiento
inference_transforms = transforms.Compose([
    transforms.Resize(256),       # Redimensionar un poco más grande
    transforms.CenterCrop(224),   # Cortar el centro exacto de 224x224
    transforms.ToTensor(),        # Convertir a números
    transforms.Normalize(         # Normalizar con medias de ImageNet
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    )
])

# 4. FUNCIÓN PARA PREDECIR LA FOTO
def clasificar_imagen(ruta_imagen, modelo):
    try:
        # Abrir imagen
        img_original = Image.open(ruta_imagen).convert('RGB')
        
        # Preprocesar
        img_tensor = inference_transforms(img_original)
        img_tensor = img_tensor.unsqueeze(0) # Añadir dimensión de batch (1, 3, 224, 224)
        img_tensor = img_tensor.to(device)
        
        # Predecir
        with torch.no_grad():
            outputs = modelo(img_tensor)
            # Obtenemos probabilidades con Softmax
            probs = torch.nn.functional.softmax(outputs, dim=1)
            confianza, prediccion = torch.max(probs, 1)
            
        clase_ganadora = CLASSES[prediccion.item()]
        probabilidad = confianza.item() * 100
        
        # Mostrar resultado
        plt.figure(figsize=(6,6))
        plt.imshow(img_original)
        plt.axis('off')
        plt.title(f"Predicción: {clase_ganadora}\nConfianza: {probabilidad:.2f}%")
        plt.show()
        
        return clase_ganadora, probabilidad

    except Exception as e:
        print(f"Error al cargar la imagen: {e}")
        return None, None

In [ ]:
# --- EJECUCIÓN ---
modelo_listo = cargar_modelo()

# CAMBIA ESTO POR LA RUTA DE TU FOTO
mi_foto = "Botella_Vidrio_Hornitos.jpg" 

# Clasificación
clase, conf = clasificar_imagen(mi_foto, modelo_listo)

In [ ]:
# --- EJECUCIÓN ---
modelo_listo = cargar_modelo()

# CAMBIA ESTO POR LA RUTA DE TU FOTO
mi_foto = "Envoltura_Gansito.jpg" 

# Clasificación
clase, conf = clasificar_imagen(mi_foto, modelo_listo)

In [ ]:
# --- EJECUCIÓN ---
modelo_listo = cargar_modelo()

# CAMBIA ESTO POR LA RUTA DE TU FOTO
mi_foto = "Lata_Volt.jpg" 

# Clasificación
clase, conf = clasificar_imagen(mi_foto, modelo_listo)

In [ ]:
# --- EJECUCIÓN ---
modelo_listo = cargar_modelo()

# CAMBIA ESTO POR LA RUTA DE TU FOTO
mi_foto = "Platos de unicel y como que unos vasos y basura en general.jpg" 

# Clasificación
clase, conf = clasificar_imagen(mi_foto, modelo_listo)

In [ ]:
# --- EJECUCIÓN ---
modelo_listo = cargar_modelo()

# CAMBIA ESTO POR LA RUTA DE TU FOTO
mi_foto = "Playera del Alan de rayas que se pone a veces para ir a la escuela.jpg " 

# Clasificación
clase, conf = clasificar_imagen(mi_foto, modelo_listo)

In [ ]:
# --- EJECUCIÓN ---
modelo_listo = cargar_modelo()

# CAMBIA ESTO POR LA RUTA DE TU FOTO
mi_foto = "Termo negroide.jpg" 

# Clasificación
clase, conf = clasificar_imagen(mi_foto, modelo_listo)

In [ ]:
# --- EJECUCIÓN ---
modelo_listo = cargar_modelo()

# CAMBIA ESTO POR LA RUTA DE TU FOTO
mi_foto = "Envoltura de un gansito, pero la mamá envoltura.jpg" 

# Clasificación
clase, conf = clasificar_imagen(mi_foto, modelo_listo)

In [ ]:
# --- EJECUCIÓN ---
modelo_listo = cargar_modelo()

# CAMBIA ESTO POR LA RUTA DE TU FOTO
mi_foto = "Luffy.jpg" 

# Clasificación
clase, conf = clasificar_imagen(mi_foto, modelo_listo)

In [ ]:
# --- EJECUCIÓN ---
modelo_listo = cargar_modelo()

# CAMBIA ESTO POR LA RUTA DE TU FOTO
mi_foto = "Aguacate.jpg" 

# Clasificación
clase, conf = clasificar_imagen(mi_foto, modelo_listo)

In [ ]:
# --- EJECUCIÓN ---
modelo_listo = cargar_modelo()

# CAMBIA ESTO POR LA RUTA DE TU FOTO
mi_foto = "Sudadera_Flash.jpg" 

# Clasificación
clase, conf = clasificar_imagen(mi_foto, modelo_listo)

In [ ]:
# --- EJECUCIÓN ---
modelo_listo = cargar_modelo()

# CAMBIA ESTO POR LA RUTA DE TU FOTO
mi_foto = "Limones en plato.jpg" 

# Clasificación
clase, conf = clasificar_imagen(mi_foto, modelo_listo)

In [ ]:
# --- EJECUCIÓN ---
modelo_listo = cargar_modelo()

# CAMBIA ESTO POR LA RUTA DE TU FOTO
mi_foto = "Vaso_Vidrio.jpg"

# Clasificación
clase, conf = clasificar_imagen(mi_foto, modelo_listo)

In [ ]:
# --- EJECUCIÓN ---
modelo_listo = cargar_modelo()

# CAMBIA ESTO POR LA RUTA DE TU FOTO
mi_foto = "Captura de pantalla 2025-12-01 233844.png"

# Clasificación
clase, conf = clasificar_imagen(mi_foto, modelo_listo)

In [ ]:
import os
import glob
import torch
import torch_directml
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt

# --- 1. CONFIGURACIÓN ---
CARPETA_PRUEBAS = "pruebas_reales"  # Asegúrate de que esta carpeta exista y tenga fotos
MODELO_PATH = "modelo_basura_resnet18_amd.pth"
CLASSES = [
    'paper', 'cardboard', 'plastic', 'metal', 'trash', 'battery',
    'shoes', 'clothes', 'green-glass', 'brown-glass', 'white-glass', 'biological'
]

# Configuración del dispositivo (AMD)
try:
    device = torch_directml.device()
except:
    device = torch.device("cpu")
print(f" Usando dispositivo para inferencia: {device}")

# --- 2. PREPARACIÓN DEL MODELO Y TRANSFORMACIONES ---

# Transformación corregida (SQUISH) para no recortar bordes importantes
inference_transforms = transforms.Compose([
    transforms.Resize((224, 224)),  # Forzamos el tamaño exacto (puede deformar un poco, pero ve todo)
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def cargar_modelo():
    print(" Cargando modelo...")
    # Creamos la arquitectura idéntica a la del entrenamiento
    model = models.resnet18(pretrained=False)
    num_ftrs = model.fc.in_features
    model.fc = torch.nn.Linear(num_ftrs, len(CLASSES))
    
    # Cargar pesos guardados
    if os.path.exists(MODELO_PATH):
        model.load_state_dict(torch.load(MODELO_PATH, map_location=device))
        model = model.to(device)
        model.eval() # IMPORTANTE: Modo evaluación
        print(" Modelo cargado correctamente.")
        return model
    else:
        print(f" Error: No se encontró el archivo '{MODELO_PATH}'")
        return None

# --- 3. PROCESAR CARPETA Y MOSTRAR RESULTADOS ---
def probar_directorio(modelo):
    # Buscar todas las imágenes comunes
    tipos = ('*.jpg', '*.jpeg', '*.png', '*.webp')
    imagenes = []
    for ext in tipos:
        imagenes.extend(glob.glob(os.path.join(CARPETA_PRUEBAS, ext)))
    
    if not imagenes:
        print(f" No hay imágenes en la carpeta '{CARPETA_PRUEBAS}'")
        return

    print(f" Procesando {len(imagenes)} imágenes...\n")

    # Configurar Grid de visualización
    cols = 3 
    rows = (len(imagenes) // cols) + 1
    plt.figure(figsize=(15, 5 * rows))

    for i, ruta_img in enumerate(imagenes):
        try:
            # A. Cargar y Preprocesar
            img_original = Image.open(ruta_img).convert('RGB')
            img_tensor = inference_transforms(img_original).unsqueeze(0).to(device)

            # B. Predecir
            with torch.no_grad():
                outputs = modelo(img_tensor)
                probs = torch.nn.functional.softmax(outputs, dim=1)
                confianza, prediccion = torch.max(probs, 1)

            clase_predicha = CLASSES[prediccion.item()]
            confianza_pct = confianza.item() * 100
            
            # C. Mostrar
            ax = plt.subplot(rows, cols, i + 1)
            ax.imshow(img_original)
            ax.axis('off')
            
            # Color del título según confianza
            color = 'green' if confianza_pct > 70 else 'red'
            nombre_archivo = os.path.basename(ruta_img)
            
            ax.set_title(f"{nombre_archivo}\nPred: {clase_predicha}\nConf: {confianza_pct:.1f}%", 
                         color=color, fontsize=11, fontweight='bold')

            print(f"{nombre_archivo}: {clase_predicha} ({confianza_pct:.1f}%)")

        except Exception as e:
            print(f" Error con {ruta_img}: {e}")

    plt.tight_layout()
    plt.show()

# --- 4. EJECUCIÓN AUTOMÁTICA ---
if __name__ == "__main__":
    if not os.path.exists(CARPETA_PRUEBAS):
        os.makedirs(CARPETA_PRUEBAS)
        print(f" Carpeta '{CARPETA_PRUEBAS}' creada. Pon tus fotos ahí y vuelve a correr.")
    else:
        mi_modelo = cargar_modelo()
        if mi_modelo:
            probar_directorio(mi_modelo)

Modelo considerando los resultados de arriba

In [ ]:
# Instalamos albumentations asegurando que NumPy no se actualice a la versión 2.0 (que rompe AMD)
%pip install albumentations "numpy<2.0"

In [ ]:
# --- BLOQUE MAESTRO: RESNET50 (PARA AMD) ---
import os
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch_directml
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from PIL import Image
from sklearn.model_selection import train_test_split

# 1. Configuración Segura
DATA_DIR = 'garbage_classification'
BATCH_SIZE = 14        # 14 para máxima estabilidad en AMD
IMG_SIZE = 224        # ResNet usa estándar 224x224
EPOCHS = 15
NUM_CLASSES = 12

# Configuración AMD
try:
    device = torch_directml.device()
    print(f" AMD Potencia Activada: {device}")
except:
    device = torch.device("cpu")
    print(" Usando CPU")

# 2. Data Augmentation (Mantenemos la robustez)
# Intentamos importar albumentations, si falla usamos transforms estándar
try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
    
    print(" Usando Albumentations para deformaciones avanzadas.")
    train_transform = A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=20, p=0.6),
        A.RandomBrightnessContrast(p=0.4),
        A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
        A.ImageCompression(quality_lower=70, quality_upper=100, p=0.3),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
    
    val_transform = A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
    
    USE_ALBUMENTATIONS = True

except ImportError:
    print(" Albumentations no detectado. Usando torchvision estándar.")
    # Fallback a torchvision si albumentations sigue dando lata
    train_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(20),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    val_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    USE_ALBUMENTATIONS = False

# 3. Dataset
class GarbageDataset(Dataset):
    def __init__(self, filepaths, labels, transform=None):
        self.filepaths = filepaths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        path = self.filepaths[idx]
        image = Image.open(path).convert("RGB")
        label = self.labels[idx]

        if USE_ALBUMENTATIONS:
            image_np = np.array(image)
            augmented = self.transform(image=image_np)
            image = augmented["image"]
        else:
            if self.transform:
                image = self.transform(image)
        
        return image, torch.tensor(label, dtype=torch.long)

# 4. Carga de Datos
filepaths = []
labels = []
classes = sorted(os.listdir(DATA_DIR))
class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}

for cls_name in classes:
    cls_folder = os.path.join(DATA_DIR, cls_name)
    if os.path.isdir(cls_folder):
        for img_name in os.listdir(cls_folder):
            filepaths.append(os.path.join(cls_folder, img_name))
            labels.append(class_to_idx[cls_name])

# Split
X_train, X_val, y_train, y_val = train_test_split(filepaths, labels, test_size=0.2, stratify=labels, random_state=42)

# Dataloaders
train_ds = GarbageDataset(X_train, y_train, transform=train_transform)
val_ds = GarbageDataset(X_val, y_val, transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, pin_memory=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=False)

# 5. MODELO: ResNet50 (El tanque de guerra)
print("Construyendo ResNet50...")
# Usamos weights=None y cargamos pretrained manualmente para máxima compatibilidad con versiones viejas
try:
    # Intento moderno
    model = models.resnet50(weights='DEFAULT')
except:
    # Intento clásico (compatible con torch < 1.13)
    model = models.resnet50(pretrained=True)

# Ajustar última capa
num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.5), # Dropout alto para evitar memorización
    nn.Linear(num_ftrs, NUM_CLASSES)
)

model = model.to(device)

# 6. Entrenamiento
# Quitamos label_smoothing para evitar errores de driver en AMD
criterion = nn.CrossEntropyLoss() 
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)

best_acc = 0.0
print(f" Iniciando entrenamiento con ResNet50 ({EPOCHS} épocas)...")

for epoch in range(EPOCHS):
    start_time = time.time()
    model.train()
    train_loss = 0
    train_correct = 0
    total_train = 0
    
    for images, targets in train_loader:
        images, targets = images.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        train_correct += (predicted == targets).sum().item()
        total_train += targets.size(0)
        
    # Validación
    model.eval()
    val_correct = 0
    total_val = 0
    with torch.no_grad():
        for images, targets in val_loader:
            images, targets = images.to(device), targets.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            val_correct += (predicted == targets).sum().item()
            total_val += targets.size(0)
    
    epoch_acc = val_correct / total_val
    train_acc = train_correct / total_train
    
    duration = time.time() - start_time
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Acc: {train_acc:.4f} | Val Acc: {epoch_acc:.4f} |  {duration:.0f}s")
    
    if epoch_acc > best_acc:
        best_acc = epoch_acc
        torch.save(model.state_dict(), "modelo_resnet50_pro.pth")
        print("  Modelo guardado.")

print(f" Finalizado. Mejor precisión: {best_acc:.4f}")

In [ ]:
# --- CONFIGURACIÓN ESTABILIZADA: RESNET50 CONGELADA (FEATURE EXTRACTION) ---
import os
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch_directml
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from PIL import Image
from sklearn.model_selection import train_test_split

# 1. Configuración
DATA_DIR = 'garbage_classification'
BATCH_SIZE = 15      # Al congelar capas, consumimos menos memoria
IMG_SIZE = 224
EPOCHS = 10           # Necesitamos menos épocas porque solo entrenamos la capa final
NUM_CLASSES = 12

# AMD Device
try:
    device = torch_directml.device()
    print(f"AMD Activada: {device}")
except:
    device = torch.device("cpu")
    print(" Usando CPU")

# 2. Data Augmentation (Mantenemos Albumentations suave)
try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
    
    train_transform = A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=20, p=0.6),
        A.RandomBrightnessContrast(p=0.3),
        A.GaussNoise(p=0.2), # Ruido suave
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
    
    val_transform = A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
    USE_ALBUMENTATIONS = True
    print(" Usando Albumentations.")

except ImportError:
    # Fallback por si acaso
    USE_ALBUMENTATIONS = False
    print(" Usando Transforms estándar.")
    train_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    val_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

# 3. Dataset
class GarbageDataset(Dataset):
    def __init__(self, filepaths, labels, transform=None):
        self.filepaths = filepaths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        path = self.filepaths[idx]
        try:
            image = Image.open(path).convert("RGB")
        except:
            # Si una imagen falla, devolvemos una negra para no romper el loop
            image = Image.new('RGB', (IMG_SIZE, IMG_SIZE))
        
        label = self.labels[idx]

        if USE_ALBUMENTATIONS:
            image_np = np.array(image)
            augmented = self.transform(image=image_np)
            image = augmented["image"]
        elif self.transform:
            image = self.transform(image)
        
        return image, torch.tensor(label, dtype=torch.long)

# 4. Carga de datos
filepaths = []
labels = []
classes = sorted(os.listdir(DATA_DIR))
class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}

for cls_name in classes:
    cls_folder = os.path.join(DATA_DIR, cls_name)
    if os.path.isdir(cls_folder):
        for img_name in os.listdir(cls_folder):
            filepaths.append(os.path.join(cls_folder, img_name))
            labels.append(class_to_idx[cls_name])

X_train, X_val, y_train, y_val = train_test_split(filepaths, labels, test_size=0.2, stratify=labels, random_state=42)

train_ds = GarbageDataset(X_train, y_train, transform=train_transform)
val_ds = GarbageDataset(X_val, y_val, transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, pin_memory=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=False)

# 5. MODELO CONGELADO (La Clave del Éxito)
print(" Preparando ResNet50 Congelada...")
try:
    model = models.resnet50(weights='DEFAULT')
except:
    model = models.resnet50(pretrained=True)

# --- CONGELAR EL CEREBRO ---
for param in model.parameters():
    param.requires_grad = False
# ---------------------------

# Solo desbloqueamos la capa final para que aprenda
num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(num_ftrs, NUM_CLASSES)
)

# Enviamos a GPU
model = model.to(device)

# Verificamos que solo entrenamos la capa final
params_to_update = []
for name, param in model.named_parameters():
    if param.requires_grad == True:
        params_to_update.append(param)

print(f" Capas a entrenar: {len(params_to_update)} (Solo el clasificador final)")

# 6. Entrenamiento
criterion = nn.CrossEntropyLoss()
# Optimizador solo ve los parámetros finales, no toda la red
optimizer = optim.Adam(params_to_update, lr=1e-3) 

best_acc = 0.0
print(f" Iniciando entrenamiento estable ({EPOCHS} épocas)...")

for epoch in range(EPOCHS):
    start_time = time.time()
    model.train()
    train_loss = 0
    train_correct = 0
    total_train = 0
    
    for images, targets in train_loader:
        images, targets = images.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        train_correct += (predicted == targets).sum().item()
        total_train += targets.size(0)
        
    # Validación
    model.eval()
    val_correct = 0
    total_val = 0
    with torch.no_grad():
        for images, targets in val_loader:
            images, targets = images.to(device), targets.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            val_correct += (predicted == targets).sum().item()
            total_val += targets.size(0)
    
    epoch_acc = val_correct / total_val
    train_acc = train_correct / total_train
    
    duration = time.time() - start_time
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Acc: {train_acc:.4f} | Val Acc: {epoch_acc:.4f} |  {duration:.0f}s")
    
    # Guardamos siempre si mejora
    if epoch_acc > best_acc:
        best_acc = epoch_acc
        torch.save(model.state_dict(), "modelo_resnet50_frozen.pth")
        print("   Mejor modelo guardado.")

print(f" Finalizado. Mejor precisión estable: {best_acc:.4f}")

In [ ]:
import os
import glob
import torch
import torch.nn as nn 
import torch_directml
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt

# --- 1. CONFIGURACIÓN ---
CARPETA_PRUEBAS = "pruebas_reales" 
MODELO_PATH = "modelo_resnet50_frozen.pth" # Modelo frozen entrenado
CLASSES = [
    'battery', 'biological', 'brown-glass', 'cardboard', 'clothes', 
    'green-glass', 'metal', 'paper', 'plastic', 'shoes', 'trash', 'white-glass'
]

# Configuración del dispositivo
try:
    device = torch_directml.device()
except:
    device = torch.device("cpu")
print(f" Listo para probar en: {device}")

# --- 2. TRANSFORMACIONES (SQUISH) ---
# Usamos Resize directo para no recortar bordes en fotos de celular
inference_transforms = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def cargar_modelo_resnet50():
    print(" Cargando ResNet50...")
    
    # 1. Crear la arquitectura BASE (igual que en el entrenamiento)
    # Nota: No necesitamos descargar pesos pretrained aquí, porque cargaremos los nuestros
    try:
        model = models.resnet50(weights=None) 
    except:
        model = models.resnet50(pretrained=False)

    # 2. Reconstruir la capa FINAL (CLAVE: Debe ser idéntica al entrenamiento)
    num_ftrs = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(num_ftrs, len(CLASSES))
    )
    
    # 3. Cargar el archivo de pesos
    if os.path.exists(MODELO_PATH):
        try:
            # map_location ayuda a cargar en CPU si la GPU está ocupada o viceversa
            pesos = torch.load(MODELO_PATH, map_location=device)
            model.load_state_dict(pesos)
            model = model.to(device)
            model.eval() # IMPORTANTE: Apaga el Dropout para predecir
            print(" Modelo cargado exitosamente.")
            return model
        except Exception as e:
            print(f" Error al cargar los pesos: {e}")
            return None
    else:
        print(f" No encontré el archivo '{MODELO_PATH}'. ¿Ya terminó de entrenar?")
        return None

# --- 3. PROCESAR FOTOS ---
def probar_directorio(modelo):
    # Buscar imágenes
    tipos = ('*.jpg', '*.jpeg', '*.png', '*.webp')
    imagenes = []
    for ext in tipos:
        imagenes.extend(glob.glob(os.path.join(CARPETA_PRUEBAS, ext)))
    
    if not imagenes:
        print(f"Carpeta '{CARPETA_PRUEBAS}' vacía o no existe.")
        return

    print(f" Probando {len(imagenes)} fotos...\n")

    cols = 3 
    rows = (len(imagenes) // cols) + 1
    plt.figure(figsize=(15, 5 * rows))

    for i, ruta_img in enumerate(imagenes):
        try:
            # Preprocesar
            img_original = Image.open(ruta_img).convert('RGB')
            img_tensor = inference_transforms(img_original).unsqueeze(0).to(device)

            # Predecir
            with torch.no_grad():
                outputs = modelo(img_tensor)
                probs = torch.nn.functional.softmax(outputs, dim=1)
                confianza, prediccion = torch.max(probs, 1)

            idx = prediccion.item()
            clase_predicha = CLASSES[idx]
            confianza_pct = confianza.item() * 100
            
            # Mostrar
            ax = plt.subplot(rows, cols, i + 1)
            ax.imshow(img_original)
            ax.axis('off')
            
            # Lógica de colores y advertencia
            color = 'green'
            texto_extra = ""
            
            if confianza_pct < 60:
                color = 'red'
                texto_extra = "\n(Dudoso...)"
            elif confianza_pct < 80:
                color = 'orange'
            
            nombre_archivo = os.path.basename(ruta_img)
            ax.set_title(f"{nombre_archivo}\nEs: {clase_predicha}\nSeguridad: {confianza_pct:.1f}%{texto_extra}", 
                         color=color, fontsize=11, fontweight='bold')

            print(f" {nombre_archivo}: {clase_predicha} ({confianza_pct:.1f}%)")

        except Exception as e:
            print(f" Error en {ruta_img}: {e}")

    plt.tight_layout()
    plt.show()

# --- EJECUTAR ---
if __name__ == "__main__":
    mi_modelo = cargar_modelo_resnet50()
    if mi_modelo:
        probar_directorio(mi_modelo)

In [ ]:
import os
import glob
import torch
import torch.nn as nn
import torch_directml
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt

# --- CONFIGURACIÓN ---
CARPETA_FOTOS = "pruebas_reales"    # Tu carpeta con fotos
ARCHIVO_MODELO = "modelo_resnet50_pro.pth" # Tu archivo guardado
CLASES = [
    'battery', 'biological', 'brown-glass', 'cardboard', 'clothes', 
    'green-glass', 'metal', 'paper', 'plastic', 'shoes', 'trash', 'white-glass'
]

# 1. Configurar AMD
try:
    device = torch_directml.device()
    print(f" Usando AMD: {device}")
except:
    device = torch.device("cpu")
    print("Usando CPU")

# 2. Transformaciones (Squish para no recortar bordes)
transformacion = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 3. Cargar Modelo
def cargar_modelo():
    print(f"Cargando {ARCHIVO_MODELO}...")
    
    # Arquitectura Base
    try:
        model = models.resnet50(weights=None)
    except:
        model = models.resnet50(pretrained=False)

    # Reconstruir la capa final EXACTA del entrenamiento (Dropout 0.5)
    num_ftrs = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(num_ftrs, len(CLASES))
    )
    
    # Cargar Pesos
    if os.path.exists(ARCHIVO_MODELO):
        state_dict = torch.load(ARCHIVO_MODELO, map_location=device)
        model.load_state_dict(state_dict)
        model = model.to(device)
        model.eval() # Modo evaluación (apaga el dropout)
        print(" Modelo listo.")
        return model
    else:
        print(f" Error: No existe el archivo {ARCHIVO_MODELO}")
        return None

# 4. Ejecutar Clasificación
def clasificar_todo():
    modelo = cargar_modelo()
    if not modelo: return

    # Buscar imágenes
    fotos = []
    for ext in ('*.jpg', '*.jpeg', '*.png', '*.webp'):
        fotos.extend(glob.glob(os.path.join(CARPETA_FOTOS, ext)))
    
    if not fotos:
        print(f" No hay fotos en '{CARPETA_FOTOS}'")
        return

    print(f" Procesando {len(fotos)} fotos...")
    
    # Configurar visualización
    cols = 3
    rows = (len(fotos) // cols) + 1
    plt.figure(figsize=(15, 5 * rows))

    for i, ruta in enumerate(fotos):
        try:
            # Procesar
            img_orig = Image.open(ruta).convert('RGB')
            img_t = transformacion(img_orig).unsqueeze(0).to(device)

            # Predecir
            with torch.no_grad():
                out = modelo(img_t)
                probs = torch.nn.functional.softmax(out, dim=1)
                conf, pred = torch.max(probs, 1)

            clase = CLASES[pred.item()]
            pct = conf.item() * 100
            
            # Mostrar
            ax = plt.subplot(rows, cols, i + 1)
            ax.imshow(img_orig)
            ax.axis('off')
            
            color = 'green' if pct > 70 else 'red'
            nombre = os.path.basename(ruta)
            ax.set_title(f"{nombre}\n{clase}\n({pct:.1f}%)", color=color, fontweight='bold')
            
            print(f" {nombre}: {clase} ({pct:.1f}%)")

        except Exception as e:
            print(f" Error en {ruta}: {e}")

    plt.tight_layout()
    plt.show()

# --- CORRER ---
if __name__ == "__main__":
    if not os.path.exists(CARPETA_FOTOS):
        os.makedirs(CARPETA_FOTOS)
        print(f" Carpeta '{CARPETA_FOTOS}' creada. Pon tus fotos ahí.")
    else:
        clasificar_todo()

In [ ]:
import torch
import torch.nn as nn
import torch_directml
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt
import os

# --- 1. CONFIGURACIÓN ---
# Pon aquí el nombre exacto de tu foto
RUTA_IMAGEN = "como-es-proceso-reciclaje-carton.jpg" 

ARCHIVO_MODELO = "modelo_resnet50_frozen.pth" # Modelo a usar
CLASES = [
    'battery', 'biological', 'brown-glass', 'cardboard', 'clothes', 
    'green-glass', 'metal', 'paper', 'plastic', 'shoes', 'trash', 'white-glass'
]

# Configurar AMD
try:
    device = torch_directml.device()
except:
    device = torch.device("cpu")

# --- 2. TRANSFORMACIÓN (Squish) ---
transformacion = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# --- 3. CARGAR MODELO ---
def cargar_modelo():
    try:
        model = models.resnet50(weights=None) # O pretrained=False
    except:
        model = models.resnet50(pretrained=False)

    # Reconstruir la capa final (IMPORTANTE: Igual al entrenamiento)
    num_ftrs = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(num_ftrs, len(CLASES))
    )
    
    if os.path.exists(ARCHIVO_MODELO):
        model.load_state_dict(torch.load(ARCHIVO_MODELO, map_location=device))
        model = model.to(device)
        model.eval()
        return model
    else:
        print(f" No encontré el archivo del modelo: {ARCHIVO_MODELO}")
        return None

# --- 4. PREDECIR UNA SOLA FOTO ---
def predecir_foto(ruta, modelo):
    if not os.path.exists(ruta):
        print(f" No encuentro la imagen: {ruta}")
        return

    try:
        # Procesar
        img_original = Image.open(ruta).convert('RGB')
        img_tensor = transformacion(img_original).unsqueeze(0).to(device) # Añadir batch

        # Inferencia
        with torch.no_grad():
            output = modelo(img_tensor)
            probs = torch.nn.functional.softmax(output, dim=1)
            confianza, indice = torch.max(probs, 1)

        clase = CLASES[indice.item()]
        pct = confianza.item() * 100

        # Mostrar Resultado
        plt.figure(figsize=(6, 6))
        plt.imshow(img_original)
        plt.axis('off')
        
        color = 'green' if pct > 70 else 'red'
        plt.title(f"Predicción: {clase}\nConfianza: {pct:.1f}%", color=color, fontsize=14, fontweight='bold')
        plt.show()
        
        print(f"Resultado: Es '{clase}' con {pct:.1f}% de seguridad.")

    except Exception as e:
        print(f" Error al procesar la imagen: {e}")

# --- EJECUTAR ---
if __name__ == "__main__":
    mi_modelo = cargar_modelo()
    if mi_modelo:
        predecir_foto(RUTA_IMAGEN, mi_modelo)

Métricas 

In [ ]:
# Instalar seaborn manteniendo la compatibilidad con tu AMD
%pip install seaborn "numpy<2.0"

In [ ]:
import os
import torch
import torch.nn as nn
import torch_directml
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from PIL import Image

# --- CONFIGURACIÓN ---
DATA_DIR = 'garbage_classification'
MODELO_PATH = "modelo_resnet50_frozen.pth" # Asegúrate que este sea el nombre correcto
BATCH_SIZE = 16
IMG_SIZE = 224
CLASSES = sorted(os.listdir(DATA_DIR)) # ['battery', 'biological', ...]

# Configurar AMD
try:
    device = torch_directml.device()
except:
    device = torch.device("cpu")
print(f" Evaluando en: {device}")

# --- RECONSTRUIR EL DATASET DE PRUEBA (VALIDACIÓN) ---
# Necesitamos exactamente el mismo split (random_state=42) para evaluar las mismas fotos
filepaths = []
labels = []
class_to_idx = {cls_name: i for i, cls_name in enumerate(CLASSES)}

for cls_name in CLASSES:
    cls_folder = os.path.join(DATA_DIR, cls_name)
    if os.path.isdir(cls_folder):
        for img_name in os.listdir(cls_folder):
            filepaths.append(os.path.join(cls_folder, img_name))
            labels.append(class_to_idx[cls_name])

# El mismo split del entrenamiento
_, X_test, _, y_test = train_test_split(filepaths, labels, test_size=0.2, stratify=labels, random_state=42)

# Dataset simple para evaluación
class TestDataset(Dataset):
    def __init__(self, filepaths, labels):
        self.filepaths = filepaths
        self.labels = labels
        self.transform = transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        img = Image.open(self.filepaths[idx]).convert("RGB")
        return self.transform(img), self.labels[idx]

test_ds = TestDataset(X_test, y_test)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# --- CARGAR MODELO ---
def cargar_modelo():
    try:
        model = models.resnet50(weights=None)
    except:
        model = models.resnet50(pretrained=False)
    
    num_ftrs = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(num_ftrs, len(CLASSES))
    )
    
    if os.path.exists(MODELO_PATH):
        model.load_state_dict(torch.load(MODELO_PATH, map_location=device))
        model = model.to(device)
        model.eval()
        return model
    else:
        print(" No encontré el archivo del modelo.")
        return None

# --- GENERAR REPORTES ---
model = cargar_modelo()

if model:
    print(" Generando predicciones... (Esto puede tardar un poco)")
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, targets in test_loader:
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(targets.numpy())

    # 1. Reporte de Clasificación (Texto)
    print("\n" + "="*60)
    print(" REPORTE DE CLASIFICACIÓN")
    print("="*60)
    print(classification_report(all_labels, all_preds, target_names=CLASSES))

    # 2. Matriz de Confusión (Gráfico)
    cm = confusion_matrix(all_labels, all_preds)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=CLASSES, yticklabels=CLASSES)
    plt.xlabel('Predicción')
    plt.ylabel('Realidad')
    plt.title('Matriz de Confusión - ResNet50 Congelada')
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score, cohen_kappa_score
from sklearn.preprocessing import label_binarize
from itertools import cycle

# --- 1. OBTENER PROBABILIDADES (Necesario para las curvas) ---
# Usamos el mismo 'test_loader' y 'model' que ya cargaste antes
def obtener_probabilidades(model, loader, device):
    model.eval()
    y_true = []
    y_scores = [] # Aquí guardaremos la probabilidad bruta

    print(" Calculando probabilidades para curvas...")
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            # Aplicamos Softmax para tener porcentajes reales (0 a 1)
            probs = torch.nn.functional.softmax(outputs, dim=1)
            
            y_true.extend(labels.numpy())
            y_scores.extend(probs.cpu().numpy())
            
    return np.array(y_true), np.array(y_scores)

# Ejecutamos la función
y_test_np, y_score_np = obtener_probabilidades(model, test_loader, device)

# Binarizar etiquetas (Convertir "clase 0" a [1, 0, 0...]) para ROC multi-clase
y_test_bin = label_binarize(y_test_np, classes=range(len(CLASSES)))
n_classes = y_test_bin.shape[1]

# --- 2. GRAFICAR CURVA ROC (Receiver Operating Characteristic) ---
# Mide qué tan bueno es separando clases positivas de negativas
fpr = dict()
tpr = dict()
roc_auc = dict()

for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_score_np[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Graficar todas las clases
plt.figure(figsize=(10, 8))
colors = cycle(['blue', 'red', 'green', 'orange', 'purple', 'cyan', 'magenta', 'yellow', 'black', 'grey', 'brown', 'pink'])
for i, color in zip(range(n_classes), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=2,
             label='ROC {0} (area = {1:0.2f})'.format(CLASSES[i], roc_auc[i]))

plt.plot([0, 1], [0, 1], 'k--', lw=2) # Línea de suerte (50%)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Tasa de Falsos Positivos')
plt.ylabel('Tasa de Verdaderos Positivos')
plt.title('Curva ROC Multi-clase')
plt.legend(loc="lower right", fontsize='small')
plt.show()

# --- 3. GRAFICAR CURVA PRECISION-RECALL ---
# Mide qué tan preciso es cuando dice "es positivo" (muy útil para clases desbalanceadas)
precision = dict()
recall = dict()
average_precision = dict()

for i in range(n_classes):
    precision[i], recall[i], _ = precision_recall_curve(y_test_bin[:, i], y_score_np[:, i])
    average_precision[i] = average_precision_score(y_test_bin[:, i], y_score_np[:, i])

plt.figure(figsize=(10, 8))
for i, color in zip(range(n_classes), colors):
    plt.plot(recall[i], precision[i], color=color, lw=2,
             label='PR {0} (AP = {1:0.2f})'.format(CLASSES[i], average_precision[i]))

plt.xlabel('Recall (Sensibilidad)')
plt.ylabel('Precision')
plt.title('Curva Precision-Recall Multi-clase')
plt.legend(loc="lower left", fontsize='small')
plt.show()

# --- 4. MÉTRICA EXTRA: COHEN'S KAPPA ---
# Dice qué tanto mejor es tu modelo que un clasificador aleatorio (0 = aleatorio, 1 = perfecto)
# Es más robusta que el "Accuracy" simple.
y_pred_clases = np.argmax(y_score_np, axis=1)
kappa = cohen_kappa_score(y_test_np, y_pred_clases)

print("="*40)
print(f" Cohen's Kappa Score: {kappa:.4f}")
print("="*40)
if kappa > 0.8:
    print("INTERPRETACIÓN: Acuerdo 'Casi Perfecto'. Tu modelo es excelente.")
elif kappa > 0.6:
    print("INTERPRETACIÓN: Acuerdo 'Sustancial'. Modelo muy sólido.")
else:
    print("INTERPRETACIÓN: Acuerdo moderado o bajo.")

In [ ]:
import os
import torch
import torch.nn as nn
import torch_directml
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve, average_precision_score, cohen_kappa_score
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from PIL import Image
from itertools import cycle

# --- 1. CONFIGURACIÓN ---
DATA_DIR = 'garbage_classification'
MODELO_PATH = "modelo_resnet50_frozen.pth"
BATCH_SIZE = 16
IMG_SIZE = 224
CLASSES = sorted(os.listdir(DATA_DIR))

# Configurar AMD
try:
    device = torch_directml.device()
    print(f"Evaluando en dispositivo: {device}")
except:
    device = torch.device("cpu")
    print("Usando CPU")

# --- 2. PREPARACIÓN DE DATOS (TEST SET) ---
# Reconstruimos el split exacto (random_state=42) para evaluar solo en datos no vistos
filepaths = []
labels = []
class_to_idx = {cls_name: i for i, cls_name in enumerate(CLASSES)}

for cls_name in CLASSES:
    cls_folder = os.path.join(DATA_DIR, cls_name)
    if os.path.isdir(cls_folder):
        for img_name in os.listdir(cls_folder):
            filepaths.append(os.path.join(cls_folder, img_name))
            labels.append(class_to_idx[cls_name])

# Separamos el 20% para prueba
_, X_test, _, y_test = train_test_split(filepaths, labels, test_size=0.2, stratify=labels, random_state=42)

class TestDataset(Dataset):
    def __init__(self, filepaths, labels):
        self.filepaths = filepaths
        self.labels = labels
        self.transform = transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        img = Image.open(self.filepaths[idx]).convert("RGB")
        return self.transform(img), self.labels[idx]

test_ds = TestDataset(X_test, y_test)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# --- 3. CARGAR MODELO ---
def cargar_modelo():
    try:
        model = models.resnet50(weights=None)
    except:
        model = models.resnet50(pretrained=False)
    
    num_ftrs = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(num_ftrs, len(CLASSES))
    )
    
    if os.path.exists(MODELO_PATH):
        model.load_state_dict(torch.load(MODELO_PATH, map_location=device))
        model = model.to(device)
        model.eval()
        return model
    else:
        print(f"Error: No se encontró {MODELO_PATH}")
        return None

# --- 4. EJECUCIÓN DE EVALUACIÓN ---
model = cargar_modelo()

if model:
    print("Calculando predicciones y probabilidades...")
    
    y_true = []
    y_pred = []
    y_score = [] # Probabilidades para curvas ROC/PR

    with torch.no_grad():
        for images, targets in test_loader:
            images = images.to(device)
            outputs = model(images)
            
            # Probabilidades
            probs = torch.nn.functional.softmax(outputs, dim=1)
            
            # Clase predicha
            _, preds = torch.max(outputs, 1)
            
            y_true.extend(targets.numpy())
            y_pred.extend(preds.cpu().numpy())
            y_score.extend(probs.cpu().numpy())

    # Convertir a numpy
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_score = np.array(y_score)

    # ---------------------------------------------------------
    # MÉTRICA 1: COHEN'S KAPPA
    # ---------------------------------------------------------
    kappa = cohen_kappa_score(y_true, y_pred)
    print("\n" + "="*50)
    print(f"COHEN'S KAPPA SCORE: {kappa:.4f}")
    if kappa > 0.8: print("(Acuerdo casi perfecto)")
    elif kappa > 0.6: print("(Acuerdo sustancial)")
    print("="*50)

    # ---------------------------------------------------------
    # MÉTRICA 2: REPORTE DE CLASIFICACIÓN
    # ---------------------------------------------------------
    print("\nREPORTE DE CLASIFICACIÓN:")
    print(classification_report(y_true, y_pred, target_names=CLASSES))

    # ---------------------------------------------------------
    # MÉTRICA 3: MATRIZ DE CONFUSIÓN
    # ---------------------------------------------------------
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES)
    plt.xlabel('Predicción')
    plt.ylabel('Realidad')
    plt.title('Matriz de Confusión')
    plt.xticks(rotation=45)
    plt.show()

    # ---------------------------------------------------------
    # PREPARACIÓN PARA CURVAS (BINARIZACIÓN)
    # ---------------------------------------------------------
    y_test_bin = label_binarize(y_true, classes=range(len(CLASSES)))
    n_classes = y_test_bin.shape[1]
    colors = cycle(['blue', 'red', 'green', 'orange', 'purple', 'brown', 'pink', 'gray', 'olive', 'cyan'])

    # ---------------------------------------------------------
    # MÉTRICA 4: CURVA ROC (Receiver Operating Characteristic)
    # ---------------------------------------------------------
    fpr = dict()
    tpr = dict()
    roc_auc = dict()

    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_score[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    plt.figure(figsize=(10, 8))
    for i, color in zip(range(n_classes), colors):
        plt.plot(fpr[i], tpr[i], color=color, lw=2,
                 label='ROC {0} (AUC = {1:0.2f})'.format(CLASSES[i], roc_auc[i]))

    plt.plot([0, 1], [0, 1], 'k--', lw=2)
    plt.xlabel('Tasa de Falsos Positivos')
    plt.ylabel('Tasa de Verdaderos Positivos')
    plt.title('Curva ROC Multi-clase')
    plt.legend(loc="lower right", fontsize='small')
    plt.grid(alpha=0.3)
    plt.show()

    # ---------------------------------------------------------
    # MÉTRICA 5: CURVA PRECISION-RECALL
    # ---------------------------------------------------------
    precision = dict()
    recall = dict()
    average_precision = dict()

    for i in range(n_classes):
        precision[i], recall[i], _ = precision_recall_curve(y_test_bin[:, i], y_score[:, i])
        average_precision[i] = average_precision_score(y_test_bin[:, i], y_score[:, i])

    plt.figure(figsize=(10, 8))
    # Reiniciar ciclo de colores
    colors = cycle(['blue', 'red', 'green', 'orange', 'purple', 'brown', 'pink', 'gray', 'olive', 'cyan'])
    
    for i, color in zip(range(n_classes), colors):
        plt.plot(recall[i], precision[i], color=color, lw=2,
                 label='PR {0} (AP = {1:0.2f})'.format(CLASSES[i], average_precision[i]))

    plt.xlabel('Recall (Sensibilidad)')
    plt.ylabel('Precision')
    plt.title('Curva Precision-Recall')
    plt.legend(loc="lower left", fontsize='small')
    plt.grid(alpha=0.3)
    plt.show()

MODELO CON CLASES BALANCEADAS.

In [ ]:
import os
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch_directml
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models, transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from collections import Counter

# --- 1. CONFIGURACIÓN ---
DATA_DIR = 'garbage_classification'
BATCH_SIZE = 16
IMG_SIZE = 224
EPOCHS = 15 # Un poco más de épocas para que aproveche el re-muestreo
NUM_CLASSES = 12

# Configurar AMD
try:
    device = torch_directml.device()
    print(f" AMD Activada: {device}")
except:
    device = torch.device("cpu")
    print(" Usando CPU")

# --- 2. DATA AUGMENTATION (RUDA) ---
try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
    
    # Entrenamiento: Transformaciones fuertes
    train_transform = A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.2), # Agregamos vertical para objetos tirados
        A.Rotate(limit=30, p=0.7),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
        A.MotionBlur(blur_limit=3, p=0.2),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
    
    # Validación: Solo normalizar (INTACTA)
    val_transform = A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
    USE_ALBUMENTATIONS = True
    print(" Albumentations configurado.")

except ImportError:
    print("Error: Necesitas albumentations instalado.")
    # (Asumimos que ya lo tienes instalado de los pasos anteriores)

# --- 3. PREPARACIÓN Y SPLIT (85/15) ---
filepaths = []
labels = []
classes = sorted(os.listdir(DATA_DIR))
class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}

print(" Leyendo archivos...")
for cls_name in classes:
    cls_folder = os.path.join(DATA_DIR, cls_name)
    if os.path.isdir(cls_folder):
        for img_name in os.listdir(cls_folder):
            filepaths.append(os.path.join(cls_folder, img_name))
            labels.append(class_to_idx[cls_name])

# División 85% Train - 15% Val
X_train, X_val, y_train, y_val = train_test_split(
    filepaths, labels, 
    test_size=0.15, # <--- AQUÍ ESTÁ EL 15%
    stratify=labels, 
    random_state=42
)

print(f" Total imágenes: {len(filepaths)}")
print(f"   - Entrenamiento (85%): {len(X_train)}")
print(f"   - Prueba Intacta (15%): {len(X_val)}")

# --- 4. LÓGICA DE BALANCEO (WEIGHTED SAMPLER) ---
# Solo aplicamos esto al set de entrenamiento
print(" Calculando pesos para balancear clases en entrenamiento...")

# Contamos cuántas imágenes hay de cada clase en el TRAIN set
class_counts = Counter(y_train)

# Calculamos el peso de cada clase (Inverso de la frecuencia)
# Si hay pocos metales, su peso será ALTO. Si hay mucha ropa, su peso será BAJO.
class_weights = []
for i in range(len(classes)):
    # +1 para evitar división por cero por seguridad
    weight = 1.0 / class_counts[i]
    class_weights.append(weight)

# Asignamos un peso a CADA IMAGEN del set de entrenamiento
sample_weights = [class_weights[label] for label in y_train]

# Creamos el Sampler
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(X_train),
    replacement=True # Importante: Permite repetir imágenes (oversampling)
)
print(" Sampler creado. El modelo verá las clases balanceadas.")

# --- 5. DATASET Y LOADERS ---
class GarbageDataset(Dataset):
    def __init__(self, filepaths, labels, transform=None):
        self.filepaths = filepaths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        path = self.filepaths[idx]
        try:
            image = Image.open(path).convert("RGB")
        except:
            image = Image.new('RGB', (IMG_SIZE, IMG_SIZE)) # Fallback imagen negra
        
        label = self.labels[idx]

        if USE_ALBUMENTATIONS:
            image_np = np.array(image)
            augmented = self.transform(image=image_np)
            image = augmented["image"]
        
        return image, torch.tensor(label, dtype=torch.long)

train_ds = GarbageDataset(X_train, y_train, transform=train_transform)
val_ds = GarbageDataset(X_val, y_val, transform=val_transform)

# IMPORTANTE: En train_loader usamos 'sampler' y QUITAMOS 'shuffle' (son incompatibles)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, pin_memory=False)
# En val_loader NO usamos sampler, queremos ver la realidad desbalanceada
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=False)

# --- 6. MODELO RESNET50 CONGELADA ---
print(" Construyendo ResNet50 Congelada...")
try:
    model = models.resnet50(weights='DEFAULT')
except:
    model = models.resnet50(pretrained=True)

# Congelar todo
for param in model.parameters():
    param.requires_grad = False

# Capa final personalizada
num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.5), # Dropout agresivo para evitar memorización
    nn.Linear(num_ftrs, NUM_CLASSES)
)

model = model.to(device)

# --- 7. ENTRENAMIENTO ---
criterion = nn.CrossEntropyLoss()
# Optimizador solo para la capa final
optimizer = optim.AdamW(model.fc.parameters(), lr=1e-3, weight_decay=1e-3)

best_acc = 0.0
print(f" Iniciando entrenamiento balanceado ({EPOCHS} épocas)...")

for epoch in range(EPOCHS):
    start_time = time.time()
    model.train()
    train_loss = 0
    train_correct = 0
    total_train = 0
    
    for images, targets in train_loader:
        images, targets = images.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        train_correct += (predicted == targets).sum().item()
        total_train += targets.size(0)
        
    # Validación (En datos reales, desbalanceados)
    model.eval()
    val_correct = 0
    total_val = 0
    with torch.no_grad():
        for images, targets in val_loader:
            images, targets = images.to(device), targets.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            val_correct += (predicted == targets).sum().item()
            total_val += targets.size(0)
    
    epoch_acc = val_correct / total_val
    train_acc = train_correct / total_train
    
    duration = time.time() - start_time
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Acc: {train_acc:.4f} | Val Acc: {epoch_acc:.4f} |  {duration:.0f}s")
    
    if epoch_acc > best_acc:
        best_acc = epoch_acc
        torch.save(model.state_dict(), "modelo_resnet50_balanceado.pth")
        print("   Mejor modelo guardado")

print(f" Entrenamiento finalizado. Mejor Precisión en Prueba: {best_acc:.4f}")

In [ ]:
import os
import glob
import torch
import torch.nn as nn
import torch_directml
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt

# --- CONFIGURACION ---
CARPETA_PRUEBAS = "pruebas_reales"
ARCHIVO_MODELO = "modelo_resnet50_balanceado.pth"
CLASES = [
    'battery', 'biological', 'brown-glass', 'cardboard', 'clothes', 
    'green-glass', 'metal', 'paper', 'plastic', 'shoes', 'trash', 'white-glass'
]

# Configurar dispositivo AMD
try:
    device = torch_directml.device()
    print(f"Dispositivo de inferencia: {device}")
except:
    device = torch.device("cpu")
    print("Usando CPU")

# Transformaciones (Resize y Normalizacion)
transformacion = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def cargar_modelo():
    print(f"Cargando modelo desde {ARCHIVO_MODELO}...")
    
    # Reconstruir arquitectura
    try:
        model = models.resnet50(weights=None)
    except:
        model = models.resnet50(pretrained=False)

    # Reconstruir capa final
    num_ftrs = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(num_ftrs, len(CLASES))
    )
    
    # Cargar pesos
    if os.path.exists(ARCHIVO_MODELO):
        try:
            state_dict = torch.load(ARCHIVO_MODELO, map_location=device)
            model.load_state_dict(state_dict)
            model = model.to(device)
            model.eval()
            print("Modelo cargado correctamente.")
            return model
        except Exception as e:
            print(f"Error al cargar los pesos: {e}")
            return None
    else:
        print(f"Error: No existe el archivo {ARCHIVO_MODELO}")
        return None

def procesar_carpeta():
    modelo = cargar_modelo()
    if not modelo:
        return

    # Buscar imagenes
    tipos = ('*.jpg', '*.jpeg', '*.png', '*.webp')
    imagenes = []
    for ext in tipos:
        imagenes.extend(glob.glob(os.path.join(CARPETA_PRUEBAS, ext)))
    
    if not imagenes:
        print(f"La carpeta '{CARPETA_PRUEBAS}' esta vacia o no existe.")
        return

    print(f"Procesando {len(imagenes)} imagenes...")

    # Configurar visualizacion
    cols = 4 
    rows = (len(imagenes) // cols) + 1
    plt.figure(figsize=(16, 5 * rows))

    for i, ruta_img in enumerate(imagenes):
        try:
            # Cargar y preprocesar
            img_original = Image.open(ruta_img).convert('RGB')
            img_tensor = transformacion(img_original).unsqueeze(0).to(device)

            # Inferencia
            with torch.no_grad():
                outputs = modelo(img_tensor)
                probs = torch.nn.functional.softmax(outputs, dim=1)
                confianza, idx = torch.max(probs, 1)

            idx = idx.item()
            clase_predicha = CLASES[idx]
            confianza_pct = confianza.item() * 100
            
            # Graficar
            ax = plt.subplot(rows, cols, i + 1)
            ax.imshow(img_original)
            ax.axis('off')
            
            # Color del titulo segun confianza
            color_texto = 'green'
            etiqueta_extra = ""
            
            if confianza_pct < 60:
                color_texto = 'red'
                etiqueta_extra = "\n(Dudoso)"
            elif confianza_pct < 80:
                color_texto = 'orange'
            
            nombre_archivo = os.path.basename(ruta_img)
            titulo = f"{nombre_archivo}\n{clase_predicha}\n{confianza_pct:.1f}%{etiqueta_extra}"
            
            ax.set_title(titulo, color=color_texto, fontsize=10, fontweight='bold')
            
            print(f"Archivo: {nombre_archivo} -> {clase_predicha} ({confianza_pct:.1f}%)")

        except Exception as e:
            print(f"Error procesando {ruta_img}: {e}")

    plt.tight_layout()
    plt.show()

# Ejecutar
if __name__ == "__main__":
    procesar_carpeta()

In [ ]:
import os
import torch
import torch.nn as nn
import torch_directml
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve, average_precision_score
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from PIL import Image
from itertools import cycle

# --- CONFIGURACION ---
DATA_DIR = 'garbage_classification'
ARCHIVO_MODELO = "modelo_resnet50_balanceado.pth"
BATCH_SIZE = 16
IMG_SIZE = 224
CLASES = sorted(os.listdir(DATA_DIR))

# Configurar dispositivo
try:
    device = torch_directml.device()
    print(f"Evaluando en: {device}")
except:
    device = torch.device("cpu")
    print("Usando CPU")

# --- PREPARACION DE DATOS (TEST SET) ---
filepaths = []
labels = []
class_to_idx = {cls_name: i for i, cls_name in enumerate(CLASES)}

for cls_name in CLASES:
    cls_folder = os.path.join(DATA_DIR, cls_name)
    if os.path.isdir(cls_folder):
        for img_name in os.listdir(cls_folder):
            filepaths.append(os.path.join(cls_folder, img_name))
            labels.append(class_to_idx[cls_name])

# IMPORTANTE: Usamos test_size=0.15 y random_state=42 para replicar el split del entrenamiento
_, X_test, _, y_test = train_test_split(
    filepaths, labels, 
    test_size=0.15, 
    stratify=labels, 
    random_state=42
)

class TestDataset(Dataset):
    def __init__(self, filepaths, labels):
        self.filepaths = filepaths
        self.labels = labels
        self.transform = transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        img = Image.open(self.filepaths[idx]).convert("RGB")
        return self.transform(img), self.labels[idx]

test_ds = TestDataset(X_test, y_test)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# --- CARGAR MODELO ---
def cargar_modelo():
    try:
        model = models.resnet50(weights=None)
    except:
        model = models.resnet50(pretrained=False)
    
    num_ftrs = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(num_ftrs, len(CLASES))
    )
    
    if os.path.exists(ARCHIVO_MODELO):
        model.load_state_dict(torch.load(ARCHIVO_MODELO, map_location=device))
        model = model.to(device)
        model.eval()
        return model
    else:
        print(f"Error: No se encontro el archivo {ARCHIVO_MODELO}")
        return None

# --- EJECUCION ---
model = cargar_modelo()

if model:
    print("Generando predicciones sobre el set de prueba...")
    
    y_true = []
    y_pred = []
    y_score = [] # Probabilidades para curvas

    with torch.no_grad():
        for images, targets in test_loader:
            images = images.to(device)
            outputs = model(images)
            
            # Probabilidades (Softmax)
            probs = torch.nn.functional.softmax(outputs, dim=1)
            
            # Clase ganadora
            _, preds = torch.max(outputs, 1)
            
            y_true.extend(targets.numpy())
            y_pred.extend(preds.cpu().numpy())
            y_score.extend(probs.cpu().numpy())

    # Convertir a numpy arrays
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_score = np.array(y_score)

    # ---------------------------------------------------------
    # 1. REPORTE DE CLASIFICACION
    # ---------------------------------------------------------
    print("\n" + "="*60)
    print("REPORTE DE CLASIFICACION")
    print("="*60)
    print(classification_report(y_true, y_pred, target_names=CLASES))

    # ---------------------------------------------------------
    # 2. MATRIZ DE CONFUSION
    # ---------------------------------------------------------
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASES, yticklabels=CLASES)
    plt.xlabel('Prediccion')
    plt.ylabel('Realidad')
    plt.title('Matriz de Confusion (Balanceada)')
    plt.xticks(rotation=45)
    plt.show()

    # Preparar datos para curvas multi-clase
    y_test_bin = label_binarize(y_true, classes=range(len(CLASES)))
    n_classes = y_test_bin.shape[1]
    colors = cycle(['blue', 'red', 'green', 'orange', 'purple', 'brown', 'pink', 'gray', 'olive', 'cyan', 'magenta', 'yellow'])

    # ---------------------------------------------------------
    # 3. CURVA ROC
    # ---------------------------------------------------------
    fpr = dict()
    tpr = dict()
    roc_auc = dict()

    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_score[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    plt.figure(figsize=(10, 8))
    for i, color in zip(range(n_classes), colors):
        plt.plot(fpr[i], tpr[i], color=color, lw=2,
                 label='ROC {0} (AUC = {1:0.2f})'.format(CLASES[i], roc_auc[i]))

    plt.plot([0, 1], [0, 1], 'k--', lw=2)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Tasa de Falsos Positivos')
    plt.ylabel('Tasa de Verdaderos Positivos')
    plt.title('Curva ROC')
    plt.legend(loc="lower right", fontsize='small')
    plt.grid(alpha=0.3)
    plt.show()

    # ---------------------------------------------------------
    # 4. CURVA PRECISION-RECALL
    # ---------------------------------------------------------
    precision = dict()
    recall = dict()
    average_precision = dict()

    for i in range(n_classes):
        precision[i], recall[i], _ = precision_recall_curve(y_test_bin[:, i], y_score[:, i])
        average_precision[i] = average_precision_score(y_test_bin[:, i], y_score[:, i])

    plt.figure(figsize=(10, 8))
    # Reiniciar colores
    colors = cycle(['blue', 'red', 'green', 'orange', 'purple', 'brown', 'pink', 'gray', 'olive', 'cyan', 'magenta', 'yellow'])
    
    for i, color in zip(range(n_classes), colors):
        plt.plot(recall[i], precision[i], color=color, lw=2,
                 label='PR {0} (AP = {1:0.2f})'.format(CLASES[i], average_precision[i]))

    plt.xlabel('Recall (Sensibilidad)')
    plt.ylabel('Precision')
    plt.title('Curva Precision-Recall')
    plt.legend(loc="lower left", fontsize='small')
    plt.grid(alpha=0.3)
    plt.show()

In [ ]:
import os
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch_directml
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models
from PIL import Image
from sklearn.model_selection import train_test_split
from collections import Counter

# Instalar albumentations si falta
try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
except ImportError:
    import sys
    os.system(f"{sys.executable} -m pip install albumentations")
    import albumentations as A
    from albumentations.pytorch import ToTensorV2

# --- CONFIGURACION ---
DATA_DIR = 'garbage_classification'
BATCH_SIZE = 16 
IMG_SIZE = 224
EPOCHS_WARMUP = 5
EPOCHS_FINETUNE = 10
NUM_CLASSES = 12

# Configuración AMD
try:
    device = torch_directml.device()
    print(f" Dispositivo activado: {device}")
except:
    device = torch.device("cpu")
    print(" Usando CPU")

# --- DATA AUGMENTATION ---
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.Rotate(limit=30, p=0.7),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.6),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
    A.MotionBlur(blur_limit=3, p=0.2),
    A.CoarseDropout(max_holes=8, max_height=20, max_width=20, p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

# --- DATOS ---
filepaths = []
labels = []
classes = sorted(os.listdir(DATA_DIR))
class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}

for cls_name in classes:
    cls_folder = os.path.join(DATA_DIR, cls_name)
    if os.path.isdir(cls_folder):
        for img_name in os.listdir(cls_folder):
            filepaths.append(os.path.join(cls_folder, img_name))
            labels.append(class_to_idx[cls_name])

# Split
X_train, X_val, y_train, y_val = train_test_split(
    filepaths, labels, test_size=0.15, stratify=labels, random_state=42
)

# Sampler (Balanceo)
class_counts = Counter(y_train)
class_weights = [1.0 / class_counts[i] for i in range(len(classes))]
sample_weights = [class_weights[label] for label in y_train]
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(X_train), replacement=True)

class GarbageDataset(Dataset):
    def __init__(self, filepaths, labels, transform=None):
        self.filepaths = filepaths
        self.labels = labels
        self.transform = transform
    def __len__(self): return len(self.filepaths)
    def __getitem__(self, idx):
        try:
            image = np.array(Image.open(self.filepaths[idx]).convert("RGB"))
        except:
            image = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        label = self.labels[idx]
        if self.transform:
            augmented = self.transform(image=image)
            image = augmented["image"]
        return image, torch.tensor(label, dtype=torch.long)

train_loader = DataLoader(GarbageDataset(X_train, y_train, train_transform), 
                          batch_size=BATCH_SIZE, sampler=sampler, pin_memory=False)
val_loader = DataLoader(GarbageDataset(X_val, y_val, val_transform), 
                        batch_size=BATCH_SIZE, shuffle=False, pin_memory=False)

# --- MODELO ---
print(" Configurando ResNet50...")
try:
    model = models.resnet50(weights='DEFAULT')
except:
    model = models.resnet50(pretrained=True)

# Congelar todo inicialmente
for param in model.parameters():
    param.requires_grad = False

num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(num_ftrs, NUM_CLASSES)
)
model = model.to(device)

# CORRECCIÓN: Quitamos label_smoothing para estabilidad en AMD
criterion = nn.CrossEntropyLoss()

# --- FASE 1: WARMUP ---
optimizer = optim.AdamW(model.fc.parameters(), lr=1e-3, weight_decay=1e-3)
best_acc = 0.0

print(f"\n FASE 1: Calentamiento ({EPOCHS_WARMUP} epocas)...")
for epoch in range(EPOCHS_WARMUP):
    model.train()
    train_correct = 0
    total_train = 0
    
    for images, targets in train_loader:
        images, targets = images.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        _, predicted = torch.max(outputs, 1)
        train_correct += (predicted == targets).sum().item()
        total_train += targets.size(0)
    
    print(f"   Epoch {epoch+1} | Train Acc: {train_correct/total_train:.4f}")

# --- FASE 2: FINE-TUNING ---
print(f"\n FASE 2: Fine-Tuning ({EPOCHS_FINETUNE} epocas)...")

# Descongelar bloque 4 y FC
for param in model.layer4.parameters():
    param.requires_grad = True
for param in model.fc.parameters():
    param.requires_grad = True

# LR bajo para ajuste fino
optimizer = optim.AdamW([
    {'params': model.layer4.parameters(), 'lr': 1e-4},
    {'params': model.fc.parameters(), 'lr': 1e-4}
], weight_decay=1e-3)

for epoch in range(EPOCHS_FINETUNE):
    start_time = time.time()
    model.train()
    train_correct = 0
    total_train = 0
    
    for images, targets in train_loader:
        images, targets = images.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        _, predicted = torch.max(outputs, 1)
        train_correct += (predicted == targets).sum().item()
        total_train += targets.size(0)

    # Validación
    model.eval()
    val_correct = 0
    total_val = 0
    with torch.no_grad():
        for images, targets in val_loader:
            images, targets = images.to(device), targets.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            val_correct += (predicted == targets).sum().item()
            total_val += targets.size(0)
    
    epoch_acc = val_correct / total_val
    duration = time.time() - start_time
    
    print(f"Epoch {epoch+1}/{EPOCHS_FINETUNE} | Train Acc: {train_correct/total_train:.4f} | Val Acc: {epoch_acc:.4f} |  {duration:.0f}s")
    
    if epoch_acc > best_acc:
        best_acc = epoch_acc
        torch.save(model.state_dict(), "modelo_final_finetuned.pth")
        print("   Modelo guardado.")

print(f" Finalizado. Mejor precisión: {best_acc:.4f}")

In [ ]:
import os
import glob
import torch
import torch.nn as nn
import torch_directml
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt

# --- CONFIGURACION ---
CARPETA_PRUEBAS = "pruebas_reales"
ARCHIVO_MODELO = "modelo_final_finetuned.pth"
CLASES = [
    'battery', 'biological', 'brown-glass', 'cardboard', 'clothes', 
    'green-glass', 'metal', 'paper', 'plastic', 'shoes', 'trash', 'white-glass'
]

try:
    device = torch_directml.device()
except:
    device = torch.device("cpu")

transformacion = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def cargar_modelo():
    print(f"Cargando {ARCHIVO_MODELO}...")
    try:
        model = models.resnet50(weights=None)
    except:
        model = models.resnet50(pretrained=False)

    num_ftrs = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(num_ftrs, len(CLASES))
    )
    
    if os.path.exists(ARCHIVO_MODELO):
        model.load_state_dict(torch.load(ARCHIVO_MODELO, map_location=device))
        model = model.to(device)
        model.eval()
        return model
    else:
        print(f"Error: No existe {ARCHIVO_MODELO}")
        return None

def ejecutar_pruebas():
    modelo = cargar_modelo()
    if not modelo: return

    imagenes = []
    for ext in ('*.jpg', '*.jpeg', '*.png', '*.webp'):
        imagenes.extend(glob.glob(os.path.join(CARPETA_PRUEBAS, ext)))
    
    if not imagenes:
        print(f"Carpeta {CARPETA_PRUEBAS} vacia.")
        return

    cols = 4
    rows = (len(imagenes) // cols) + 1
    plt.figure(figsize=(16, 5 * rows))

    for i, ruta in enumerate(imagenes):
        try:
            img = Image.open(ruta).convert('RGB')
            tensor = transformacion(img).unsqueeze(0).to(device)

            with torch.no_grad():
                out = modelo(tensor)
                probs = torch.nn.functional.softmax(out, dim=1)
                conf, idx = torch.max(probs, 1)

            clase = CLASES[idx.item()]
            pct = conf.item() * 100
            
            ax = plt.subplot(rows, cols, i + 1)
            ax.imshow(img)
            ax.axis('off')
            
            color = 'green' if pct > 70 else 'red'
            ax.set_title(f"{os.path.basename(ruta)}\n{clase} ({pct:.1f}%)", 
                         color=color, fontsize=10, fontweight='bold')
            print(f"{os.path.basename(ruta)} -> {clase} ({pct:.1f}%)")

        except Exception as e:
            print(f"Error en {ruta}: {e}")

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    ejecutar_pruebas()

In [ ]:
import os
import torch
import torch.nn as nn
import torch_directml
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve, average_precision_score, cohen_kappa_score
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from PIL import Image
from itertools import cycle

# --- CONFIGURACION ---
DATA_DIR = 'garbage_classification'
ARCHIVO_MODELO = "modelo_final_finetuned.pth"
BATCH_SIZE = 16
IMG_SIZE = 224
CLASES = sorted(os.listdir(DATA_DIR))

try:
    device = torch_directml.device()
except:
    device = torch.device("cpu")

# --- RECONSTRUIR TEST SET ---
filepaths = []
labels = []
class_to_idx = {cls_name: i for i, cls_name in enumerate(CLASES)}

for cls_name in CLASES:
    cls_folder = os.path.join(DATA_DIR, cls_name)
    if os.path.isdir(cls_folder):
        for img_name in os.listdir(cls_folder):
            filepaths.append(os.path.join(cls_folder, img_name))
            labels.append(class_to_idx[cls_name])

_, X_test, _, y_test = train_test_split(
    filepaths, labels, test_size=0.15, stratify=labels, random_state=42
)

class TestDataset(Dataset):
    def __init__(self, filepaths, labels):
        self.filepaths = filepaths
        self.labels = labels
        self.transform = transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
    def __len__(self): return len(self.filepaths)
    def __getitem__(self, idx):
        img = Image.open(self.filepaths[idx]).convert("RGB")
        return self.transform(img), self.labels[idx]

test_loader = DataLoader(TestDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

# --- CARGAR MODELO ---
try:
    model = models.resnet50(weights=None)
except:
    model = models.resnet50(pretrained=False)

num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(num_ftrs, len(CLASES))
)

if os.path.exists(ARCHIVO_MODELO):
    model.load_state_dict(torch.load(ARCHIVO_MODELO, map_location=device))
    model = model.to(device)
    model.eval()
else:
    print(f"Error: No se encontro {ARCHIVO_MODELO}")
    model = None

# --- GENERAR METRICAS ---
if model:
    print("Calculando metricas...")
    y_true, y_pred, y_score = [], [], []

    with torch.no_grad():
        for images, targets in test_loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.nn.functional.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
            
            y_true.extend(targets.numpy())
            y_pred.extend(preds.cpu().numpy())
            y_score.extend(probs.cpu().numpy())

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_score = np.array(y_score)

    # Reporte
    print("\nREPORTE DE CLASIFICACION:")
    print(classification_report(y_true, y_pred, target_names=CLASES))

    # Kappa
    kappa = cohen_kappa_score(y_true, y_pred)
    print(f"COHEN'S KAPPA: {kappa:.4f}")

    # Matriz Confusion
    plt.figure(figsize=(10, 8))
    sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt='d', cmap='Blues', 
                xticklabels=CLASES, yticklabels=CLASES)
    plt.title('Matriz de Confusion')
    plt.show()

    # Curvas
    y_test_bin = label_binarize(y_true, classes=range(len(CLASES)))
    n_classes = y_test_bin.shape[1]
    colors = cycle(['blue', 'red', 'green', 'orange', 'purple', 'brown', 'pink', 'gray', 'olive', 'cyan', 'magenta', 'yellow'])

    # ROC
    plt.figure(figsize=(10, 8))
    for i, color in zip(range(n_classes), colors):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
        plt.plot(fpr, tpr, color=color, lw=2, label=f'ROC {CLASES[i]} (AUC = {auc(fpr, tpr):.2f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.title('Curva ROC')
    plt.legend(loc="lower right", fontsize='small')
    plt.show()

    # Precision-Recall
    plt.figure(figsize=(10, 8))
    for i, color in zip(range(n_classes), colors):
        precision, recall, _ = precision_recall_curve(y_test_bin[:, i], y_score[:, i])
        ap = average_precision_score(y_test_bin[:, i], y_score[:, i])
        plt.plot(recall, precision, color=color, lw=2, label=f'PR {CLASES[i]} (AP = {ap:.2f})')
    plt.title('Curva Precision-Recall')
    plt.legend(loc="lower left", fontsize='small')
    plt.show()

In [ ]:
import os
import torch
import torch.nn as nn
import torch_directml
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from torchvision import models, transforms
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from PIL import Image
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve, average_precision_score, cohen_kappa_score
from sklearn.preprocessing import label_binarize
from collections import Counter
from itertools import cycle

# --- CONFIGURACION ---
DATA_DIR = 'garbage_classification'
BATCH_SIZE = 32
IMG_SIZE = 224
CLASES = sorted(os.listdir(DATA_DIR))
MODELO_RESNET = "resnet50_extractor.pth"
MODELO_LDA = "lda_classifier.joblib"

try:
    device = torch_directml.device()
    print(f"Dispositivo: {device}")
except:
    device = torch.device("cpu")

# --- PREPARACION DE DATOS ---
filepaths = []
labels = []
class_to_idx = {cls_name: i for i, cls_name in enumerate(CLASES)}

for cls_name in CLASES:
    cls_folder = os.path.join(DATA_DIR, cls_name)
    if os.path.isdir(cls_folder):
        for img_name in os.listdir(cls_folder):
            filepaths.append(os.path.join(cls_folder, img_name))
            labels.append(class_to_idx[cls_name])

# Split 85/15
X_train, X_test, y_train, y_test = train_test_split(
    filepaths, labels, test_size=0.15, stratify=labels, random_state=42
)

# Sampler para balancear el entrenamiento (Importante para LDA)
class_counts = Counter(y_train)
class_weights = [1.0 / class_counts[i] for i in range(len(CLASES))]
sample_weights = [class_weights[label] for label in y_train]
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(X_train), replacement=True)

class FeatureDataset(Dataset):
    def __init__(self, filepaths, labels):
        self.filepaths = filepaths
        self.labels = labels
        self.transform = transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
    def __len__(self): return len(self.filepaths)
    def __getitem__(self, idx):
        try:
            img = Image.open(self.filepaths[idx]).convert("RGB")
        except:
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE))
        return self.transform(img), self.labels[idx]

# Dataloaders
train_loader = DataLoader(FeatureDataset(X_train, y_train), batch_size=BATCH_SIZE, sampler=sampler)
test_loader = DataLoader(FeatureDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

# --- MODELO EXTRACTOR (ResNet50 sin capa final) ---
print("Preparando extractor de caracteristicas...")
model = models.resnet50(weights='DEFAULT')
# Quitamos la capa final (fc) reemplazandola por Identidad
model.fc = nn.Identity()
model = model.to(device)
model.eval() # Siempre en modo eval para extraer

def extraer_features(loader, model, device):
    features_list = []
    labels_list = []
    print("Extrayendo caracteristicas...")
    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device)
            # Salida: [Batch, 2048]
            embeddings = model(images)
            features_list.append(embeddings.cpu().numpy())
            labels_list.append(targets.numpy())
    return np.vstack(features_list), np.concatenate(labels_list)

# 1. Extraer vectores
X_train_feats, y_train_labels = extraer_features(train_loader, model, device)
X_test_feats, y_test_labels = extraer_features(test_loader, model, device)

# 2. Entrenar LDA
print(f"Entrenando LDA con {X_train_feats.shape[0]} muestras...")
lda = LinearDiscriminantAnalysis()
lda.fit(X_train_feats, y_train_labels)

# 3. Guardar Modelos
torch.save(model.state_dict(), MODELO_RESNET)
joblib.dump(lda, MODELO_LDA)
print("Modelos guardados (Extractor + LDA).")

# --- EVALUACION ---
print("Evaluando...")
y_pred = lda.predict(X_test_feats)
y_prob = lda.predict_proba(X_test_feats)

# Reporte
print("\nREPORTE DE CLASIFICACION (LDA):")
print(classification_report(y_test_labels, y_pred, target_names=CLASES))

# Kappa
kappa = cohen_kappa_score(y_test_labels, y_pred)
print(f"COHEN'S KAPPA: {kappa:.4f}")

# Matriz
plt.figure(figsize=(10, 8))
sns.heatmap(confusion_matrix(y_test_labels, y_pred), annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASES, yticklabels=CLASES)
plt.title('Matriz de Confusion (ResNet + LDA)')
plt.show()

# Curvas
y_test_bin = label_binarize(y_test_labels, classes=range(len(CLASES)))
n_classes = y_test_bin.shape[1]
colors = cycle(['blue', 'red', 'green', 'orange', 'purple', 'brown', 'pink', 'gray', 'olive', 'cyan', 'magenta', 'yellow'])

# ROC
plt.figure(figsize=(10, 8))
for i, color in zip(range(n_classes), colors):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_prob[:, i])
    plt.plot(fpr, tpr, color=color, lw=2, label=f'ROC {CLASES[i]} (AUC = {auc(fpr, tpr):.2f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.title('Curva ROC (LDA)')
plt.legend(loc="lower right", fontsize='small')
plt.show()

# Precision-Recall
plt.figure(figsize=(10, 8))
for i, color in zip(range(n_classes), colors):
    precision, recall, _ = precision_recall_curve(y_test_bin[:, i], y_prob[:, i])
    ap = average_precision_score(y_test_bin[:, i], y_prob[:, i])
    plt.plot(recall, precision, color=color, lw=2, label=f'PR {CLASES[i]} (AP = {ap:.2f})')
plt.title('Curva Precision-Recall (LDA)')
plt.legend(loc="lower left", fontsize='small')
plt.show()

In [ ]:
import os
import glob
import torch
import torch.nn as nn
import torch_directml
import joblib
import matplotlib.pyplot as plt
from torchvision import models, transforms
from PIL import Image

# --- CONFIGURACION ---
CARPETA_PRUEBAS = "pruebas_reales"
MODELO_RESNET = "resnet50_extractor.pth"
MODELO_LDA = "lda_classifier.joblib"
CLASES = [
    'battery', 'biological', 'brown-glass', 'cardboard', 'clothes', 
    'green-glass', 'metal', 'paper', 'plastic', 'shoes', 'trash', 'white-glass'
]

try:
    device = torch_directml.device()
except:
    device = torch.device("cpu")

transformacion = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def cargar_sistema():
    print("Cargando sistema hibrido...")
    
    # 1. Cargar ResNet Extractor
    if not os.path.exists(MODELO_RESNET):
        print("Falta el modelo ResNet")
        return None, None
        
    model = models.resnet50(weights=None)
    model.fc = nn.Identity() # Importante: Quitar capa final
    model.load_state_dict(torch.load(MODELO_RESNET, map_location=device))
    model = model.to(device)
    model.eval()
    
    # 2. Cargar LDA
    if not os.path.exists(MODELO_LDA):
        print("Falta el modelo LDA")
        return None, None
        
    lda = joblib.load(MODELO_LDA)
    
    return model, lda

def ejecutar_pruebas():
    extractor, clasificador = cargar_sistema()
    if not extractor or not clasificador: return

    imagenes = []
    for ext in ('*.jpg', '*.jpeg', '*.png', '*.webp'):
        imagenes.extend(glob.glob(os.path.join(CARPETA_PRUEBAS, ext)))
    
    if not imagenes:
        print("Carpeta vacia.")
        return

    cols = 4
    rows = (len(imagenes) // cols) + 1
    plt.figure(figsize=(16, 5 * rows))

    print(f"Procesando {len(imagenes)} imagenes con LDA...")

    for i, ruta in enumerate(imagenes):
        try:
            img = Image.open(ruta).convert('RGB')
            tensor = transformacion(img).unsqueeze(0).to(device)

            # Paso 1: Extraer vector (Deep Learning)
            with torch.no_grad():
                feature_vector = extractor(tensor).cpu().numpy()
            
            # Paso 2: Clasificar vector (Machine Learning Clasico)
            probs = clasificador.predict_proba(feature_vector)[0]
            pred_idx = clasificador.predict(feature_vector)[0]
            
            clase = CLASES[pred_idx]
            confianza = probs[pred_idx] * 100
            
            # Visualizar
            ax = plt.subplot(rows, cols, i + 1)
            ax.imshow(img)
            ax.axis('off')
            
            color = 'green' if confianza > 70 else 'red'
            ax.set_title(f"{os.path.basename(ruta)}\n{clase} ({confianza:.1f}%)", 
                         color=color, fontsize=10, fontweight='bold')
            
            print(f"{os.path.basename(ruta)} -> {clase} ({confianza:.1f}%)")

        except Exception as e:
            print(f"Error en {ruta}: {e}")

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    ejecutar_pruebas()

In [ ]:
import pkg_resources

# Lista completa de librerías para Entrenamiento + Inferencia
librerias = [
    'torch', 
    'torchvision', 
    'scikit-learn', 
    'numpy', 
    'pandas', 
    'matplotlib', 
    'Pillow',
    'joblib'
]

print("Copia y pega esto en tu requirements.txt:\n")
for lib in librerias:
    try:
        version = pkg_resources.get_distribution(lib).version
        print(f"{lib}=={version}")
    except pkg_resources.DistributionNotFound:
        print(f"# No se pudo obtener la versión de {lib}")